> 在线讲义：[第 1 章 · 一个成功率有多准：30 次试验里成了 21 次，真实水平在哪](https://wikieden.github.io/math-and-eval-stats/lessons/01-success-rate-ci.html) · 本书《数学与实验统计》

# 第 1 章 · 一个成功率有多准

讲义：[01-success-rate-ci.md](https://wikieden.github.io/math-and-eval-stats/lessons/01-success-rate-ci.html)。一格一格往下跑；只算讲义里讲过的例子，练习的数自己算。

## 讲义 ↔ 代码

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 | p̂ = k/n | `point_estimate(21, 30)` | 0.70 |
| 第二步 | SE、Wald | `standard_error / wald_ci` | 0.0837、[0.536, 0.864] |
| 第三步 | Wilson | `wilson_ci(21, 30)` | [0.521, 0.833] |
| 第四步 | n ≈ z²p(1−p)/h² | `n_for_half_width(0.7, 0.05)` | 323 |
| 第五步 | 覆盖率 | `coverage(wald_ci / wilson_ci, 0.9, 30)` | 0.809 / 0.974 |

## 定义

In [ ]:
import math
import numpy as np
Z95 = 1.959964  # 标准正态分布 97.5% 分位数，95% 双侧区间用它（常写成 1.96）

`point_estimate`：第一步：成功率的点估计 p̂ = k / n。

In [ ]:
def point_estimate(k: int, n: int) -> float:
    """第一步：成功率的点估计 p̂ = k / n。"""
    return k / n

`standard_error`：第二步：标准误 SE = sqrt(p̂(1-p̂)/n)。

In [ ]:
def standard_error(k: int, n: int) -> float:
    """第二步：标准误 SE = sqrt(p̂(1-p̂)/n)。"""
    p = k / n
    return math.sqrt(p * (1 - p) / n)

`wald_ci`：第二步：Wald 区间 p̂ ± z·SE（教科书最常见、小样本最不靠谱的那个）。

In [ ]:
def wald_ci(k: int, n: int, z: float = Z95) -> tuple[float, float]:
    """第二步：Wald 区间 p̂ ± z·SE（教科书最常见、小样本最不靠谱的那个）。"""
    p, se = k / n, standard_error(k, n)
    return max(0.0, p - z * se), min(1.0, p + z * se)

`wilson_ci`：第三步：Wilson 区间。中心往 0.5 拉一点，宽度里多一项 z²/4n²，所以 p̂=0 或 1 时也不会缩成一个点。

In [ ]:
def wilson_ci(k: int, n: int, z: float = Z95) -> tuple[float, float]:
    """第三步：Wilson 区间。中心往 0.5 拉一点，宽度里多一项 z²/4n²，所以 p̂=0 或 1 时也不会缩成一个点。"""
    p, z2 = k / n, z * z
    denom = 1 + z2 / n
    center = (p + z2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / denom
    return center - half, center + half

`n_for_half_width`：第四步：想让区间半宽 ≤ half_width，大约要多少次试验（按 Wald 公式反解，向上取整）。

In [ ]:
def n_for_half_width(p: float, half_width: float, z: float = Z95) -> int:
    """第四步：想让区间半宽 ≤ half_width，大约要多少次试验（按 Wald 公式反解，向上取整）。"""
    return math.ceil(z * z * p * (1 - p) / half_width**2)

`coverage`：第五步：重复做 reps 次「测 n 次」的实验，数一数区间盖住真值 p_true 的比例。

In [ ]:
def coverage(method, p_true: float, n: int, reps: int = 20000, seed: int = 0) -> float:
    """第五步：重复做 reps 次「测 n 次」的实验，数一数区间盖住真值 p_true 的比例。"""
    rng = np.random.default_rng(seed)
    ks = rng.binomial(n, p_true, size=reps)
    hit = 0
    for k in ks:
        lo, hi = method(int(k), n)
        hit += lo <= p_true <= hi
    return hit / reps

## 一步一步跑

### 第一步 点估计

In [ ]:
point_estimate(21, 30)

### 第二步 标准误与 Wald 区间

In [ ]:
standard_error(21, 30), wald_ci(21, 30), wald_ci(30, 30)

### 第三步 Wilson 区间

In [ ]:
wilson_ci(21, 30), wilson_ci(30, 30)

### 第四步 区间宽度随 n 缩小

In [ ]:
[(n, round((wilson_ci(round(0.7*n), n)[1]-wilson_ci(round(0.7*n), n)[0])/2*100, 1)) for n in [10, 30, 100, 300, 1000]], n_for_half_width(0.7, 0.05)

### 第五步 覆盖率模拟（真值 0.9，n = 30）

In [ ]:
coverage(wald_ci, 0.9, 30), coverage(wilson_ci, 0.9, 30)

## 代码里才看得到的

1. `z` 用 1.959964，所以和 scipy 对到小数点后 6 位。
2. `wald_ci` 把区间截在 [0, 1] 里；Wilson 天然不出界。
3. 覆盖率是模拟值，换种子第三位小数会抖。

## 自己改一改

- 把 coverage 的真值改成 0.5，看 Wald 和 Wilson 差距变小没有
- n = 10、真值 0.95，看 Wald 覆盖率掉到多少

In [ ]:
# 在这里改参数试试


## 口诀

> 一勺汤不是一锅汤，七成背后正负十五；
> 靠零靠一 Wald 塌，Wilson 往中拉一拉；
> 想窄一半测四倍，九五说的是方法。